# 이진 분류 실험 ⑩ — 못 잡은 패턴 시도 보기 (HTML)
09 의 테스트 점수로 **어떤 패턴 시도(attempt) 를 놓쳤는지**, 시도 안에서 **어느 거래만 빠졌는지** 를 본다. 학습은 없다 (점수 파일만 읽는다).

- **시도** = `Patterns.txt` 의 `BEGIN … END` 한 묶음 (`patterns.parquet` 의 `attempt_id`). 평가 구간(테스트 2주) 에 거래가 하나라도 있는 시도만 본다
- **잡음 / 놓침** = 그 거래의 점수가 알림 기준 임계값 이상인가. 기준은 09 와 같다 (테스트 전체 상위 0.1% · 0.5% · 1%, 확률 0.5)
- **시도 상태** = 평가 구간 거래 기준으로 `전부 잡음` / `일부 놓침` / `전부 놓침`
- 시도가 학습 · 검증 구간에서 시작했으면 그 거래는 점수가 없다 → `구간 밖` 으로 따로 표시 (잡음 · 놓침 계산에서 뺀다)
- 무패턴 세탁은 시도 단위가 없어서 여기에 안 나온다

**출력**: `results/HI-Large/gnn_structure/missed_patterns.html` — 조건 · 알림 기준 · 패턴 · 상태를 화면에서 바꿔 가며 본다. 파일 하나에 데이터가 다 들어 있어 내려받아 브라우저로 열면 된다
(JupyterLab 에서 바로 열면 스크립트가 막혀 빈 화면이다. 위쪽 `Trust HTML` 을 누르거나 내려받아서 연다)

필요한 파일: 이 노트북 옆의 `missed_patterns_template.html`, 09 가 만든 `<조건>_test_scores.parquet`, `parquets/HI-Large/patterns.parquet` · `transactions.parquet`

## 0. 작업 준비

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))
import bc_lib as B
L = B.L

In [ ]:
# ---- 설정 ----------------------------------------------------------------
RUN_NAME = 'gnn_structure'                                   # 점수 파일이 있는 결과 폴더 (09)
cfg = B.Config(dataset='HI-Large')
OUT_DIR = cfg.out_dir(RUN_NAME)
CONDS = [c for c in ['plain', 'rev', 'rev_port', 'rev_port_ego'] if (OUT_DIR / f'{c}_test_scores.parquet').exists()]
CUTS = {'상위 0.1%': 0.001, '상위 0.5%': 0.005, '상위 1%': 0.01, '확률 0.5': None}    # 알림 기준. 숫자 = 테스트 전체 중 상위 비율, None = 확률 0.5
DEFAULT_COND, DEFAULT_CUT = 'rev_port_ego', '상위 0.1%'       # HTML 을 열었을 때 처음 보이는 조건 · 기준 (아래 확인 표도 이 기준)
TEMPLATE = Path.cwd() / 'missed_patterns_template.html'
HTML_PATH = OUT_DIR / 'missed_patterns.html'
assert CONDS, f'점수 파일이 없다: {OUT_DIR}'
assert TEMPLATE.exists(), f'템플릿이 없다: {TEMPLATE}'
if DEFAULT_COND not in CONDS:
    DEFAULT_COND = CONDS[-1]
CONDS, OUT_DIR

## 1. 점수 · 임계값
조건마다 테스트 구간 전체 점수를 읽고, 알림 기준별 임계값을 구한다 (09 의 7 절과 같은 계산: `점수 >= 분위수`)

In [ ]:
scores = {c: B.load_scores(OUT_DIR / f'{c}_test_scores.parquet') for c in CONDS}
ref_ids = scores[CONDS[0]]['edge_id'].to_numpy()
assert (np.diff(ref_ids) > 0).all(), '점수 파일이 edge_id 순이 아니다'
for c in CONDS:
    assert np.array_equal(scores[c]['edge_id'].to_numpy(), ref_ids), f'{c} 점수의 행 순서가 다르다'
score_mat = np.column_stack([scores[c]['score'].to_numpy().astype(np.float32) for c in CONDS])     # 행 = 테스트 거래, 열 = 조건
y_test = scores[CONDS[0]]['y'].to_numpy()
del scores
score_mat.shape, int(y_test.sum())

In [ ]:
# 임계값과 그 기준의 알림 수. 동점이 많으면 알림 수가 예산보다 커진다 (09 의 plain @1% 처럼) — 여기서 확인
thr = np.array([[0.5 if b is None else float(np.quantile(score_mat[:, ci], 1 - b)) for b in CUTS.values()] for ci in range(len(CONDS))])
alerts = np.array([[int((score_mat[:, ci] >= thr[ci, ki]).sum()) for ki in range(len(CUTS))] for ci in range(len(CONDS))])
display(pd.DataFrame(thr, index=CONDS, columns=list(CUTS)).round(4).rename_axis('임계값'))
pd.DataFrame(alerts, index=CONDS, columns=list(CUTS)).rename_axis('알림 수')

## 2. 평가 구간에 걸친 패턴 시도
`patterns.parquet` 에서 평가 구간 거래가 하나라도 있는 시도를 고르고, 그 시도의 거래를 **전부** (구간 밖 포함) 가져온다

In [ ]:
pat = pd.read_parquet(L.BASE / 'patterns.parquet', columns=['attempt_id', 'pattern_type', 'pattern_desc', 'edge_id'])
pat = pat.dropna(subset=['edge_id']).astype({'edge_id': 'int64'}).drop_duplicates(['attempt_id', 'edge_id'])
pos = np.minimum(np.searchsorted(ref_ids, pat['edge_id'].to_numpy()), len(ref_ids) - 1)
pat['in_test'] = ref_ids[pos] == pat['edge_id'].to_numpy()          # 이 거래가 평가 구간에 있나 (= 점수가 있나)
pat['row'] = np.where(pat['in_test'], pos, -1)                       # 점수 행 번호
keep = pat.groupby('attempt_id')['in_test'].transform('any')
pe = pat[keep].copy()
print(f"전체 시도 {pat['attempt_id'].nunique():,}개 · 평가 구간에 걸친 시도 {pe['attempt_id'].nunique():,}개 · 그 거래 {len(pe):,}건 (구간 안 {int(pe['in_test'].sum()):,} · 구간 밖 {int((~pe['in_test']).sum()):,})")
dup = int(pe.loc[pe['in_test'], 'edge_id'].duplicated().sum())
print(f'두 시도에 같이 들어 있는 평가 구간 거래 {dup}건 (시도마다 따로 센다)')

In [ ]:
# 거래 내용 (시각 · 계좌 · 금액 · 통화 · 결제방식) — 필요한 edge_id 만 transactions.parquet 에서
con = L.connect(threads=8, memory='6GB'); t0 = time.time()
con.register('want', pd.DataFrame({'edge_id': np.unique(pe['edge_id'].to_numpy())}))
det = con.execute(f'''
    SELECT t.edge_id, t.ts, t.src, t.dst, t.received, t.receiving_currency, t.payment_format,
           t.from_bank, t.from_account, t.to_bank, t.to_account, t.is_laundering::INT AS y
    FROM read_parquet({L.q(cfg.tx_path)}) t WHERE t.edge_id IN (SELECT edge_id FROM want)''').df()
con.close()
pe = pe.merge(det, on='edge_id', how='left')
assert pe['ts'].notna().all(), 'transactions.parquet 에 없는 패턴 거래가 있다'
assert (pe['y'] == 1).all(), '패턴 거래인데 라벨이 0 인 것이 있다'
pe = pe.sort_values(['attempt_id', 'ts', 'edge_id']).reset_index(drop=True)
print(f'{time.time() - t0:.0f}s'); pe.head()

In [ ]:
# 거래마다 조건별 점수와 "잡혔나" 를 붙인다. hit[i, 조건, 기준] — 구간 밖 거래는 전부 False
rows = pe['row'].to_numpy(); in_test = pe['in_test'].to_numpy()
sc = np.full((len(pe), len(CONDS)), np.nan, np.float32); sc[in_test] = score_mat[rows[in_test]]
hit = np.zeros((len(pe), len(CONDS), len(CUTS)), bool)
hit[in_test] = sc[in_test][:, :, None] >= thr[None, :, :]
# 패턴 거래 recall 이 09 의 표와 맞는지 확인 (조건 × 기준). 두 시도에 겹친 거래는 한 번만
uniq = in_test & ~pe['edge_id'].duplicated().to_numpy()
pd.DataFrame(hit[uniq].mean(0), index=CONDS, columns=list(CUTS)).round(4).rename_axis('패턴 recall')

## 3. 시도 단위로 — 어떤 시도를 놓쳤나
평가 구간 거래 기준으로 시도마다 `전부 잡음` / `일부 놓침` / `전부 놓침` 을 매긴다

In [ ]:
def attempt_table(ci, ki):
    # 조건 ci · 기준 ki 에서 시도마다: 전체 거래 수, 평가 구간 거래 수, 잡은 수, 놓친 수, 상태
    g = pd.DataFrame({'attempt_id': pe['attempt_id'], 'pattern_type': pe['pattern_type'], 'n': 1, 'n_test': in_test.astype(int), 'n_hit': hit[:, ci, ki].astype(int)})
    a = g.groupby(['attempt_id', 'pattern_type'], as_index=False).sum()
    a['n_miss'] = a['n_test'] - a['n_hit']
    a['status'] = np.where(a['n_miss'] == 0, '전부 잡음', np.where(a['n_hit'] == 0, '전부 놓침', '일부 놓침'))
    return a


ci0, ki0 = CONDS.index(DEFAULT_COND), list(CUTS).index(DEFAULT_CUT)
att = attempt_table(ci0, ki0)
order_p = [p for p in B.PATTERN_ORDER if p in set(att['pattern_type'])]
tab = att.pivot_table(index='pattern_type', columns='status', values='attempt_id', aggfunc='count', fill_value=0).reindex(index=order_p, columns=['전부 잡음', '일부 놓침', '전부 놓침'], fill_value=0)
tab['놓친 거래'] = att.groupby('pattern_type')['n_miss'].sum(); tab['패턴 거래'] = att.groupby('pattern_type')['n_test'].sum()
tab.loc['합계'] = tab.sum()
tab['놓친 비율'] = (tab['놓친 거래'] / tab['패턴 거래']).round(4)
print(f'{DEFAULT_COND} · {DEFAULT_CUT}'); tab

In [ ]:
# 조건별로 — 놓친 거래가 있는 시도 수 (같은 기준). 장치를 켜면 "전부 놓침" 이 줄어드나, "일부 놓침" 이 줄어드나
rows_ = []
for ci, c in enumerate(CONDS):
    a = attempt_table(ci, ki0)
    rows_.append({'조건': c, '시도': len(a), **a['status'].value_counts().reindex(['전부 잡음', '일부 놓침', '전부 놓침'], fill_value=0).to_dict(),
                  '놓친 거래': int(a['n_miss'].sum()), '패턴 거래(시도 기준)': int(a['n_test'].sum())})
by_cond = pd.DataFrame(rows_).set_index('조건')
by_cond.to_csv(OUT_DIR / 'missed_attempts_by_cond.csv', encoding='utf-8-sig')
print(DEFAULT_CUT); by_cond

In [ ]:
# 일부만 놓친 시도에서, 놓친 거래는 시도의 몇 번째 거래인가 (시간순, 평가 구간 안에서 센 순서). 앞쪽 거래를 주로 놓치면 "패턴이 쌓이기 전" 이라 못 잡는 것
part_ids = set(att.loc[att['status'] == '일부 놓침', 'attempt_id'])
p = pe[in_test & pe['attempt_id'].isin(part_ids).to_numpy()].copy()
p['miss'] = ~hit[p.index.to_numpy(), ci0, ki0]
p['순서'] = p.groupby('attempt_id').cumcount() + 1; p['시도 크기'] = p.groupby('attempt_id')['edge_id'].transform('size')
p['위치'] = np.where(p['순서'] == 1, '① 첫 거래', np.where(p['순서'] == p['시도 크기'], '③ 마지막 거래', '② 중간'))
p.groupby('위치')['miss'].agg(거래='size', 놓침='sum', 놓친_비율='mean').round(4)

In [ ]:
# 시도 목록 저장 (기본 조건 · 기준) — 놓친 거래가 많은 순
att.sort_values(['n_miss', 'attempt_id'], ascending=[False, True]).to_csv(OUT_DIR / 'missed_attempts.csv', index=False, encoding='utf-8-sig')
att[att['status'] != '전부 잡음'].sort_values(['n_miss', 'attempt_id'], ascending=[False, True]).head(15)

## 4. HTML 만들기
시도 · 거래 · 조건별 점수를 JSON 으로 묶어 템플릿(`missed_patterns_template.html`) 에 넣는다. 거래 한 줄 = `[edge_id, src, dst, 시각, 금액, 통화, 결제방식, 구간 안 여부, 조건별 점수, 조건별 기준 비트]`

In [ ]:
# 계좌 이름 (은행/계좌번호) — 그림에는 A, B, C… 로 쓰고 표에 같이 적는다
nodes = {}
for node, bank, acct in zip(pe['src'], pe['from_bank'], pe['from_account']): nodes[str(int(node))] = f'{bank}/{acct}'
for node, bank, acct in zip(pe['dst'], pe['to_bank'], pe['to_account']): nodes[str(int(node))] = f'{bank}/{acct}'

bits = (hit * (1 << np.arange(len(CUTS)))[None, None, :]).sum(2).astype(int)       # 기준별 잡힘 여부를 비트로 (조건마다 정수 하나)
ts_str = pe['ts'].dt.strftime('%Y-%m-%d %H:%M').to_numpy()
cols = [pe[c].to_numpy() for c in ('edge_id', 'src', 'dst', 'received', 'receiving_currency', 'payment_format')]
attempts = []
for (aid, ptype), idx in pe.groupby(['attempt_id', 'pattern_type'], sort=True).indices.items():
    desc = pe['pattern_desc'].iat[idx[0]]
    edges = [[int(cols[0][i]), int(cols[1][i]), int(cols[2][i]), ts_str[i], round(float(cols[3][i]), 2), str(cols[4][i]), str(cols[5][i]), int(in_test[i]),
              [round(float(v), 4) for v in sc[i]] if in_test[i] else None, [int(v) for v in bits[i]] if in_test[i] else None] for i in idx]
    attempts.append(dict(id=int(aid), type=str(ptype), desc='' if pd.isna(desc) else str(desc), e=edges))
len(attempts), len(nodes)

In [ ]:
meta = dict(dataset=cfg.dataset, run=RUN_NAME, window=cfg.test_window.replace(':', ' ~ '), conds=CONDS, cuts=list(CUTS), defCond=DEFAULT_COND, defCut=DEFAULT_CUT,
            thr=[[float(v) for v in r] for r in thr], alerts=[[int(v) for v in r] for r in alerts], nTest=int(len(ref_ids)),
            typeOrder=B.PATTERN_ORDER, generated=time.strftime('%Y-%m-%d %H:%M'))
payload = json.dumps(dict(meta=meta, nodes=nodes, attempts=attempts), ensure_ascii=False, separators=(',', ':')).replace('</', '<\\/')
html = TEMPLATE.read_text(encoding='utf-8')
assert html.count('__DATA__') == 1, '템플릿에 __DATA__ 자리가 하나여야 한다'
HTML_PATH.write_text(html.replace('__DATA__', payload), encoding='utf-8')
print(HTML_PATH, f'{HTML_PATH.stat().st_size / 1e6:.1f} MB · 시도 {len(attempts):,}개')

## 5. 보는 법
- 처음 화면은 **놓친 거래가 있는 시도만** 놓친 거래가 많은 순으로 나온다. 왼쪽에서 시도를 고르면 오른쪽에 그 시도의 거래 그래프와 표가 뜬다
- 그림: 빨간 굵은 선 + `✕` = 놓친 거래, 파란 선 = 잡은 거래, 회색 점선 = 평가 구간 밖. 선 위 숫자는 시간순 거래 번호 (표의 `#`)
- 그림 위의 **보기** 단추로 바꾼다: `구조` (계좌 = 원, 거래 = 화살표) / `시간축 가로` / `시간축 세로`. 시간축 보기는 계좌가 줄이고, 거래는 그 시각에 송신 계좌 줄에서 수신 계좌 줄로 가는 화살표다
- 시간축 보기에서 거래가 몇 분 사이에 몰려 겹치면 `순서대로` 로 바꾼다 (같은 간격으로 펴고, 날짜가 바뀌는 자리만 표시). 점선 = 평가 구간의 시작 · 끝
- 표의 조건 열: 조건마다 그 거래의 점수와 ○ ✕. **`plain` 에서 ✕ 인데 `rev` 부터 ○ 인 거래** 가 그 장치가 살린 거래다
- 위쪽 표의 `일부 놓침` · `전부 놓침` 숫자를 누르면 그 패턴 · 상태로 걸러진다
- 조건 · 알림 기준을 바꾸면 전부 다시 계산된다. 방향키 ↑ ↓ 로 시도를 넘긴다